# Does rank recover the subtype and receptor information that per-sample normalisation removes? (Danenberg + Jackson IMC)

**Follow-up to `RankUtilityClinical`.** There, per-image z (per-sample) normalisation lowered patient-level AUCs relative to cohort-wide z. Rank on per-image z closed part of that gap in Jackson. Two things were still open: whether this holds across k, and whether the earlier "Jackson only" reading is right.

**Headline number (fixed before the run):** pooled % of the normalisation gap that rank closes,
  Σ_t (AUC rank_img − AUC int_img) / Σ_t (AUC int_coh − AUC int_img),
summed over the subtype and receptor targets t, with a 95% CI from 1000 paired patient-bootstrap resamples. Each resample recomputes all three AUCs.
- Danenberg targets: PAM50 + claudin-low, IntClust, ER, PR, HER2.
- Jackson targets: clinical type, ER, PR, HER2.
- Grade is reported per target but kept out of the pooled number.

**Features:** the same as `RankUtilityClinical` (same seeds, folds, C, k-means settings), with k = 10, 20, 40, 80. There is also a cluster-free version (k = 0 in the outputs): each patient's mean encoded vector, i.e. the pseudo-bulk of each encoding.
- If rank restores sample-level level information, the gain should also appear without clusters.
- If rank only makes clusters more comparable across patients, the gain should appear with clusters only.

**Decision rule (fixed before the run):** in a cohort, the claim "rank recovers sample-level subtype and receptor information" holds only if:
- the pooled CI lower bound is above 0 for at least 3 of the 4 k values;
- and it is also above 0 for the cluster-free features.

**Check:** the k = 20 and 40 AUCs should reproduce `RankUtilityClinical` exactly, since the seeds and folds are the same.

**Version 2 additions:**
- Triple-negative vs rest (`tnbc`), defined the same way in both cohorts:
  - Danenberg: ER, PR and HER2 all negative;
  - Jackson: clinical type "TripleNeg".

  It is reported per target and is not part of the pre-registered pooled number.
- rank − intensity under cohort-wide z (`delta_coh`, with CI) for every target.
- All version 1 numbers must reproduce exactly.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings
import numpy as np, pandas as pd
import pyarrow.parquet as pq
from joblib import Parallel, delayed
from sklearn.cluster import MiniBatchKMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, kruskal
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 20260929
OUT = "/kaggle/working"
COFACTOR = 1.0
MIN_CELLS = 200             # per image, as in earlier runs
KS_ALL, KS_TUM = (20, 40), (10, 20)
SEEDS = (0, 1, 2)
N_FIT = 200_000
N_REP, N_FOLD = 5, 5
C_LOGIT = 0.1
N_PCA, COX_PEN = 10, 0.1
N_BOOT, N_BOOT_HR = 1000, 300
MIN_TUM = 50                # tumour cells per patient for entropy
MIN_CLASS = 10              # patients per class; smaller classes are dropped
N_JOBS = os.cpu_count()
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
SPACES = ("int_img", "rank_img", "int_coh", "rank_coh")
PAIRS = [("int_img", "rank_img"), ("int_coh", "rank_coh")]

def num(s):
    return pd.to_numeric(pd.Series(s).replace({"nan": np.nan, "[]": np.nan}), errors="coerce")

def locate():
    reps = sorted(glob.glob("/kaggle/input/**/prep_report.json", recursive=True))
    out = {}
    for p in sorted(glob.glob("/kaggle/input/**/cells.parquet", recursive=True)):
        key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
        assert key not in out, (key, p, out)
        rep = max(reps, key=lambda r: len(os.path.commonpath([r, p])))
        clin = max(glob.glob("/kaggle/input/**/clinical.parquet", recursive=True),
                   key=lambda c: len(os.path.commonpath([c, p])))
        out[key] = dict(cells=p, report=rep, clinical=clin)
        if key == "danenberg":
            out[key]["split"] = max(glob.glob("/kaggle/input/**/danenberg_patient_split.csv", recursive=True),
                                    key=lambda c: len(os.path.commonpath([c, p])))
    print(json.dumps(out, indent=1))
    assert set(out) == {"danenberg", "jackson"}
    return out

## Encodings and clinical tables (unchanged from RankUtilityClinical)

In [ ]:
def rank_vec(Zs):
    """0 = highest; ties by column order (stable sort), as in the CyTOF code."""
    order = np.argsort(-Zs, axis=1, kind="stable")
    R = np.empty_like(order)
    R[np.arange(len(Zs))[:, None], order] = np.arange(Zs.shape[1])
    return R.astype(np.float32)

def robust_z(A):
    med = np.median(A, 0)
    mad = 1.4826 * np.median(np.abs(A - med), 0)
    sd = A.std(0)
    scale = np.where(mad > 1e-3, mad, sd + 1e-6)      # cohort-level fallback for markers with MAD = 0
    return np.clip((A - med) / scale, -5, 5).astype(np.float32)

def load(key, P):
    report = json.load(open(P["report"]))
    cells = pd.read_parquet(P["cells"])
    if key == "danenberg":
        chan = list(report["channel_map"].values())
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid, n_prot = "metabric_id", 36
    else:
        chan = list(report["channel_map"])
        cells = cells[cells.is_tumour == 1]
        pid, n_prot = "patient_id", 32
    protein = [c for c in chan if c not in ("histone_h3", "dna1", "dna2")]
    assert len(protein) == n_prot
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    epi = (cells.is_epithelial.values == 1) if key == "danenberg" else (cells.broad_class.values == "epithelial")
    img_code = pd.factorize(cells.image_name, sort=True)[0]
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    A = np.arcsinh(cells[protein].values.astype(np.float64) / COFACTOR)
    Z_img = np.empty_like(A, dtype=np.float32)
    for a, b in zip(starts[:-1], starts[1:]):             # identical to the earlier runs
        med = np.median(A[a:b], 0)
        mad = np.median(np.abs(A[a:b] - med), 0)
        Z_img[a:b] = np.clip((A[a:b] - med) / (1.4826 * mad + 1e-6), -5, 5)
    Z_coh = robust_z(A)
    spaces = {"int_img": Z_img, "rank_img": rank_vec(Z_img), "int_coh": Z_coh, "rank_coh": rank_vec(Z_coh)}
    patient = cells[pid].astype(str).values
    print(f"{key}: {len(cells):,} cells, {len(starts) - 1} images, {len(np.unique(patient))} patients, "
          f"{epi.mean():.2f} epithelial")
    return dict(protein=protein, A=A.astype(np.float32), spaces=spaces, patient=patient, epi=epi,
                phen=cells.cell_phenotype.astype(str).values)

def clinical_danenberg(P):
    c = pd.read_parquet(P["clinical"]).replace({"nan": np.nan})
    c["pid"] = c.metabric_id.astype(str)
    c = c.set_index("pid")
    split = pd.read_csv(P["split"]).set_index("metabric_id")
    t = pd.DataFrame(index=c.index)
    t["pam50"] = c.CLAUDIN_SUBTYPE.fillna(pd.Series(c.index, index=c.index).map(split.pam50))
    t["intclust"] = c.INTCLUST
    t["er"], t["pr"], t["her2"] = c.ER_STATUS, c.PR_STATUS, c.HER2_STATUS
    g = num(c.GRADE).values
    t["grade3"] = np.where(np.isnan(g), None, np.where(g == 3, "3", "1-2"))
    ln = num(c.LYMPH_NODES_EXAMINED_POSITIVE).values
    t["ln_pos"] = np.where(np.isnan(ln), None, np.where(ln > 0, "pos", "neg"))
    vs = c.VITAL_STATUS
    t["dss_time"] = num(c.OS_MONTHS).values
    t["dss_event"] = np.where(vs.isna(), np.nan, (vs == "Died of Disease").astype(float))
    rs = c.RFS_STATUS
    t["rfs_time"] = num(c.RFS_MONTHS).values
    t["rfs_event"] = np.where(rs.isna(), np.nan, rs.astype(str).str.startswith("1").astype(float))
    cov = pd.DataFrame(index=c.index)
    cov["age"] = num(c.AGE_AT_DIAGNOSIS).values
    cov["grade"] = g
    cov["log_size"] = np.log1p(num(c.TUMOR_SIZE).values)
    cov["log_nodes"] = np.log1p(ln)
    cov["er_pos"] = np.where(c.ER_STATUS.isna(), np.nan, (c.ER_STATUS == "Positive").astype(float))
    return t, cov, {"dss": ("dss_time", "dss_event"), "rfs": ("rfs_time", "rfs_event")}

def clinical_jackson(P):
    c = pd.read_parquet(P["clinical"]).replace({"nan": np.nan, "[]": np.nan})
    c = c.groupby("patient_id").first()
    c.index = c.index.astype(str)
    t = pd.DataFrame(index=c.index)
    t["clinical_type"] = c.clinical_type
    g = num(c.grade).values                                # METASTASIS -> NaN
    t["grade3"] = np.where(np.isnan(g), None, np.where(g == 3, "3", "1-2"))
    t["er"], t["pr"], t["her2"] = c.ERStatus, c.PRStatus, c.HER2Status
    basel = (c.cohort == "basel").values
    st = c.Patientstatus
    ok = basel & st.notna().values
    t["dss_time"] = np.where(basel, num(c.OSmonth).values, np.nan)
    t["dss_event"] = np.where(ok, (st == "death by primary disease").astype(float), np.nan)
    t["os_time"] = t.dss_time
    t["os_event"] = np.where(ok, st.isin(["death", "death by primary disease"]).astype(float), np.nan)
    cov = pd.DataFrame(index=c.index)
    cov["age"] = num(c.age).values
    cov["grade"] = g
    cov["log_size"] = np.log1p(num(c.tumor_size).values)
    n = c.PTNM_N.astype(str).str.strip().str.lower().str.lstrip("n")      # Basel "0", "1a"; Zurich "n0", "n1"
    cov["n_pos"] = np.where(n.str.startswith("0"), 0.0, np.where(n.str.match(r"^[123]"), 1.0, np.nan))
    return t, cov, {"dss": ("dss_time", "dss_event"), "os": ("os_time", "os_event")}

## Load both cohorts

In [ ]:
PATHS = locate()
DATA, CLIN = {}, {}
for key in ("danenberg", "jackson"):
    DATA[key] = load(key, PATHS[key])
    CLIN[key] = (clinical_danenberg if key == "danenberg" else clinical_jackson)(PATHS[key])
    t = CLIN[key][0]
    pats = np.unique(DATA[key]["patient"])
    print(key, "patients with clinical:", t.index.isin(pats).sum(), "of", len(pats))
    print(t.reindex(pats).notna().sum().to_string())

## Version 2: triple-negative vs rest, harmonised across cohorts (reported per target, not pooled)

In [ ]:
def add_tnbc(key):
    """Triple-negative vs rest, defined the same way in both cohorts (version 2)."""
    t = CLIN[key][0]
    if key == "danenberg":
        rec = t[["er", "pr", "her2"]]
        known, tn = rec.notna().all(1), (rec == "Negative").all(1)
    else:
        known, tn = t.clinical_type.notna(), t.clinical_type == "TripleNeg"
    t["tnbc"] = np.where(known, np.where(tn, "TN", "other"), None)
    s = t.reindex(np.unique(DATA[key]["patient"])).tnbc
    print(f"{key}: triple-negative {int((s == 'TN').sum())} of {int(s.notna().sum())} analysed patients")

for key in DATA:
    add_tnbc(key)

## Patient features: cell-state composition for k = 10, 20, 40, 80, and cluster-free means

In [ ]:
def shares(labels, pat_code, n_pat, k):
    M = np.zeros((n_pat, k))
    ok = labels >= 0
    np.add.at(M, (pat_code[ok], labels[ok]), 1)
    return M

def clr(M, eps=0.5):
    L = np.log(M + eps)                    # pseudo-count on counts, then share-invariant CLR
    return L - L.mean(1, keepdims=True)

def entropy_rows(M):
    p = M / np.maximum(M.sum(1, keepdims=True), 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        h = -(np.where(p > 0, p * np.log(p), 0)).sum(1)
    return h

def fit_states(V, k, seed, idx):
    rng = np.random.default_rng(seed)
    fit_idx = rng.choice(idx, min(N_FIT, len(idx)), replace=False)
    km = MiniBatchKMeans(k, random_state=seed, batch_size=4096, n_init=3, max_iter=200).fit(V[fit_idx])
    lab = np.full(len(V), -1)
    lab[idx] = km.predict(V[idx])
    return lab

def patient_specificity(lab, pat_code, k):
    """Share of cells that sit in states where a single patient contributes > 50% of the cells."""
    ok = lab >= 0
    ct = pd.crosstab(lab[ok], pat_code[ok])
    top = ct.max(1) / ct.sum(1)
    private = top.index[top > 0.5]
    eff = np.exp(entropy_rows(ct.values))                  # effective number of patients per state
    return float(np.isin(lab[ok], private).mean()), float(np.median(eff))


KS = (10, 20, 40, 80)
SP = ("int_img", "rank_img", "int_coh", "rank_coh")
FEAT = {}
t0 = time.time()
for key, D in DATA.items():
    pats, pat_code = np.unique(D["patient"], return_inverse=True)
    n_pat, idx = len(pats), np.arange(len(pat_code))
    FEAT[key] = {"pats": pats}
    for sp in SP:
        V = D["spaces"][sp]
        for k in KS:
            for s in SEEDS:
                lab = fit_states(V, k, s, idx)
                FEAT[key][(sp, k, s)] = clr(shares(lab, pat_code, n_pat, k))
        FEAT[key][(sp, 0, 0)] = pd.DataFrame(V).groupby(pat_code).mean().values      # cluster-free
        print(f"{key} {sp}: {(time.time() - t0) / 60:.1f} min")
    ph_code, ph_names = pd.factorize(D["phen"])
    FEAT[key][("phenotypes", -1, 0)] = clr(shares(ph_code, pat_code, n_pat, len(ph_names)))
    FEAT[key][("pseudobulk", -1, 0)] = pd.DataFrame(D["A"]).groupby(pat_code).mean().values

## Out-of-fold predictions and AUCs

In [ ]:
def groups_of(key):
    """Feature-set groups: (space, k) -> list of seed keys; reference sets have a single key."""
    g = {}
    for fk in FEAT[key]:
        if isinstance(fk, tuple):
            g.setdefault(fk[:2], []).append(fk)
    return g

def auc(y, P, classes):
    if len(classes) == 2:
        return roc_auc_score(y == classes[1], P[:, 1])
    return roc_auc_score(y, P, multi_class="ovr", average="macro", labels=classes)

def oof_class(X, y):
    classes = np.unique(y)
    P = np.zeros((len(y), len(classes)))
    for r in range(N_REP):
        for tr, te in StratifiedKFold(N_FOLD, shuffle=True, random_state=SEED + r).split(X, y):
            m = make_pipeline(StandardScaler(), LogisticRegression(C=C_LOGIT, max_iter=5000))
            m.fit(X[tr], y[tr])
            P[te] += m.predict_proba(X[te])
    return P / N_REP

def boot_pair(fn, n, *arrs):
    r = np.random.default_rng(SEED)
    m = len(arrs[0]); out = []
    for _ in range(n):
        i = r.integers(0, m, m)
        try:
            out.append(fn(*[a[i] for a in arrs]))
        except ValueError:
            continue
    return np.percentile(out, [2.5, 97.5]) if out else (np.nan, np.nan)


TARGETS = {"danenberg": ["pam50", "intclust", "er", "pr", "her2", "grade3", "tnbc"],
           "jackson": ["clinical_type", "er", "pr", "her2", "grade3", "tnbc"]}
POOLED = {"danenberg": ["pam50", "intclust", "er", "pr", "her2"],
          "jackson": ["clinical_type", "er", "pr", "her2"]}
YS, jobs, meta = {}, [], []
for key in DATA:
    pats = FEAT[key]["pats"]
    t = CLIN[key][0].reindex(pats)
    for tg in TARGETS[key]:
        y = t[tg].astype(object).where(t[tg].notna(), None).values
        keep = np.array([v is not None for v in y])
        vc = pd.Series(y[keep]).value_counts()
        keep &= np.isin(y, vc.index[vc >= MIN_CLASS])
        yk = np.array([str(v) for v in y[keep]])
        if len(np.unique(yk)) < 2:
            print(f"skip {key} {tg}")
            continue
        YS[(key, tg)] = (keep, yk, np.unique(yk))
        for fk in FEAT[key]:
            if isinstance(fk, tuple):
                jobs.append(delayed(oof_class)(FEAT[key][fk][keep], yk))
                meta.append((key, tg, fk))
t0 = time.time()
res = Parallel(n_jobs=N_JOBS, verbose=1)(jobs)
print(f"classification: {(time.time() - t0) / 60:.1f} min, {len(jobs)} feature sets")
PRED = {}
for (key, tg, fk), P in zip(meta, res):
    PRED.setdefault((key, tg, fk[0], fk[1]), []).append(P)
PRED = {k_: np.mean(v, 0) for k_, v in PRED.items()}          # average over k-means seeds
rows = []
for (key, tg, sp, k), P in PRED.items():
    keep, yk, classes = YS[(key, tg)]
    rows.append(dict(cohort=key, target=tg, space=sp, k=k, auc=auc(yk, P, classes), n=len(yk), n_classes=len(classes)))
aucs = pd.DataFrame(rows)
aucs.to_csv(f"{OUT}/recovery_auc.csv", index=False)
store = {}
for (key, tg, sp, k), P in PRED.items():
    store[f"{key}|{tg}|{sp}|{k}"] = P
for (key, tg), (keep, yk, classes) in YS.items():
    store[f"{key}|{tg}|y"] = yk
    store[f"{key}|{tg}|patients"] = FEAT[key]["pats"][keep]
np.savez_compressed(f"{OUT}/oof_predictions.npz", **store)
print(aucs.pivot_table(index=["cohort", "target"], columns=["space", "k"], values="auc").round(3).to_string())

## Gap closed by rank: per target and pooled, paired patient bootstrap

In [ ]:
def gap_stats(key, k, idx=None):
    """Per-target and pooled (rank_img - int_img) and (int_coh - int_img) AUC differences on a patient resample."""
    out = {}
    for tg in TARGETS[key]:
        if (key, tg) not in YS:
            continue
        keep, yk, classes = YS[(key, tg)]
        pos = np.full(len(keep), -1); pos[keep] = np.arange(keep.sum())
        i = np.arange(keep.sum()) if idx is None else pos[idx][pos[idx] >= 0]
        y = yk[i]
        if len(np.unique(y)) < len(classes):
            return None
        out[tg] = tuple(auc(y, PRED[(key, tg, sp, k)][i], classes) for sp in ("int_img", "rank_img", "int_coh", "rank_coh"))
    d = sum(out[t][1] - out[t][0] for t in POOLED[key] if t in out)
    g = sum(out[t][2] - out[t][0] for t in POOLED[key] if t in out)
    out["pooled"] = (d, g)
    return out

def boot_gap(key, k):
    n_pat = len(FEAT[key]["pats"])
    r = np.random.default_rng(SEED)
    draws = []
    for _ in range(N_BOOT):
        o = gap_stats(key, k, r.integers(0, n_pat, n_pat))
        if o is not None:
            draws.append(o)
    return key, k, gap_stats(key, k), draws

t0 = time.time()
res = Parallel(n_jobs=N_JOBS, verbose=1)(delayed(boot_gap)(key, k) for key in DATA for k in (0,) + KS)
print(f"bootstrap: {(time.time() - t0) / 60:.1f} min")
pct = lambda v: np.percentile(v, [2.5, 97.5])
rows = []
for key, k, est, draws in res:
    for tg, v in est.items():
        if tg == "pooled":
            d, g = v
            bd = np.array([x["pooled"][0] for x in draws]); bg = np.array([x["pooled"][1] for x in draws])
            row = dict(cohort=key, k=k, target="pooled", delta=d, gap=g, recovered=d / g)
        else:
            a_int, a_rank, a_ceil, a_rcoh = v
            bd = np.array([x[tg][1] - x[tg][0] for x in draws]); bg = np.array([x[tg][2] - x[tg][0] for x in draws])
            bc = np.array([x[tg][3] - x[tg][2] for x in draws])
            row = dict(cohort=key, k=k, target=tg, auc_int_img=a_int, auc_rank_img=a_rank, auc_int_coh=a_ceil,
                       auc_rank_coh=a_rcoh, delta=a_rank - a_int, gap=a_ceil - a_int,
                       recovered=(a_rank - a_int) / (a_ceil - a_int), in_pooled=tg in POOLED[key],
                       delta_coh=a_rcoh - a_ceil, delta_coh_lo=np.percentile(bc, 2.5), delta_coh_hi=np.percentile(bc, 97.5))
        row["delta_lo"], row["delta_hi"] = pct(bd)
        row["gap_lo"], row["gap_hi"] = pct(bg)
        row["recovered_lo"], row["recovered_hi"] = pct(bd / bg)
        row["n_boot"] = len(draws)
        rows.append(row)
gap = pd.DataFrame(rows).sort_values(["cohort", "target", "k"])
gap.to_csv(f"{OUT}/recovery_gap_closed.csv", index=False)
print(gap[gap.target == "pooled"].round(3).to_string(index=False))
print(gap[gap.target != "pooled"][["cohort", "target", "k", "auc_int_img", "auc_rank_img", "auc_int_coh", "delta",
                                   "delta_lo", "delta_hi", "recovered"]].round(3).to_string(index=False))

ans = {}
for key in DATA:
    p = gap[(gap.cohort == key) & (gap.target == "pooled")].set_index("k")
    n_k = int((p.loc[list(KS), "recovered_lo"] > 0).sum())
    ans[key] = {"pooled_recovered_by_k": {int(k): [float(p.loc[k, c]) for c in ("recovered", "recovered_lo", "recovered_hi")]
                                          for k in p.index},
                "k_values_with_CI_above_0": n_k,
                "cluster_free_CI_above_0": bool(p.loc[0, "recovered_lo"] > 0),
                "claim_holds": bool(n_k >= 3 and p.loc[0, "recovered_lo"] > 0)}
json.dump(ans, open(f"{OUT}/summary_recovery.json", "w"), indent=1)
print(json.dumps(ans, indent=1))